# ansel-denoise — test training on Google Colab

Trains the CFA-domain denoising U-Net on the shard cache published on the
[`shards-v1` release](https://github.com/aurelienpierreeng/ansel-denoise/releases/tag/shards-v1),
with noise synthesized on the fly from the darktable/Ansel noise profiles.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Free-tier sessions die after a few hours: checkpoints go to your Google Drive
(if you allow the mount below) and training **auto-resumes from the latest
checkpoint** — just run all cells again after a disconnect.

In [ ]:
# --- parameters -----------------------------------------------------------
STEPS = 3000        # ~30-60 min on a T4; raise for longer sessions
BATCH = 32
PATCH = 128
RUN_NAME = "colab-test"
USE_DRIVE = True    # persist checkpoints to Google Drive (asks for consent)

!nvidia-smi -L || echo 'NO GPU: switch the runtime type to T4 GPU'

In [ ]:
# --- code -----------------------------------------------------------------
# No pip install: `pip install -e .` is not importable on Colab without a
# kernel restart (the editable-install .pth finder is only processed at
# interpreter startup), and all dependencies (numpy, torch) are preinstalled
# anyway — putting src/ on sys.path is enough for this pure-Python package.
import os, sys
if not os.path.isdir('ansel-denoise'):
    !git clone --depth 1 https://github.com/aurelienpierreeng/ansel-denoise.git
%cd ansel-denoise
sys.path.insert(0, os.path.abspath('src'))
import ansel_denoise
print('package importable:', ansel_denoise.__file__)

In [ ]:
# --- data: fetch the published shard cache (no auth, public release) ------
import json, tarfile, urllib.request
from pathlib import Path

SHARDS = Path('shards/rpu')
SHARDS.mkdir(parents=True, exist_ok=True)
api = 'https://api.github.com/repos/aurelienpierreeng/ansel-denoise/releases/tags/shards-v1'
with urllib.request.urlopen(api) as r:
    assets = json.load(r)['assets']
for a in assets:
    if not a['name'].startswith('shards-'):
        continue
    print(a['name'], f"{a['size'] / 1e6:.0f} MB")
    tmp, _ = urllib.request.urlretrieve(a['browser_download_url'])
    with tarfile.open(tmp) as tar:
        tar.extractall(SHARDS, filter='data')
    Path(tmp).unlink()
print(f"{len(list(SHARDS.glob('*.npz')))} shards ready")

In [ ]:
# --- checkpoints: Google Drive if allowed, else session-local -------------
from pathlib import Path

out_root = Path('runs')
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        out_root = Path('/content/drive/MyDrive/ansel-denoise-runs')
    except Exception as e:
        print(f'Drive not mounted ({e}); checkpoints stay in the session')
OUT = out_root / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

ckpts = sorted(OUT.glob('ckpt-*.pt'))
RESUME = ['--resume', str(ckpts[-1])] if ckpts else []
print(f'checkpoints -> {OUT}', f'| resuming from {ckpts[-1].name}' if ckpts else '| fresh run')

In [ ]:
# --- train ----------------------------------------------------------------
import sys
from ansel_denoise.train import main

main(['--shards', 'shards/rpu', '--out', str(OUT),
      '--steps', str(STEPS), '--batch', str(BATCH), '--patch', str(PATCH),
      '--workers', '2', '--val-every', '500', '--ckpt-every', '500',
      *RESUME])

In [ ]:
# --- export the weights for Ansel and download them -----------------------
from ansel_denoise.export import main as export_main
export_main([str(OUT / 'ckpt-final.pt'), '--out', str(OUT / 'final.anseldn')])
try:
    from google.colab import files
    files.download(str(OUT / 'final.anseldn'))
except Exception:
    print(f'weights at {OUT}/final.anseldn')